In [1]:
import os
from typing import List

import pandas as pd


# =========================
# 1. Load Compustat data
# =========================

def load_compustat_data(path: str) -> pd.DataFrame:
    """
    Load raw Compustat data from a CSV file.
    The file is read as text first; numeric conversion happens later.
    """
    df = pd.read_csv(path, dtype=str)
    return df


# =========================
# 2. Preprocess the dataset
# =========================

def preprocess_compustat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep relevant Compustat fields, convert them to numeric formats,
    and EXPLICITLY keep the bankruptcy indicator 'dlrsn'.
    """

    # Columns to keep (we keep dlrsn here!)
    useful: List[str] = [
        "gvkey", "datadate", "fyear", "conm", "tic",
        "sic",      # industry code
        "fyr",      # fiscal year-end month

        "act", "lct", "at", "lt",
        "seq", "teq", "ceq",
        "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp",
        "ebit", "ebitda", "xint",
        "capx", "oancf", "fincf", "ivncf", "wcapch",
        "re", "pstk", "opincar",

        "csho", "mkvalt", "prcc_f", "bkvlps",

        "nipfc",

        "dlrsn",   # delisting reason (bankruptcy codes etc.)
    ]

    # Keep only columns that exist in the file
    cols_exist = [c for c in useful if c in df.columns]
    df = df[cols_exist]

    # Convert datadate → datetime
    if "datadate" in df.columns:
        df["datadate"] = pd.to_datetime(df["datadate"], errors="coerce")

    # Convert fiscal year
    if "fyear" in df.columns:
        df["fyear"] = pd.to_numeric(df["fyear"], errors="coerce").astype("Int64")

    # Convert fiscal year-end month (optional, mais propre)
    if "fyr" in df.columns:
        df["fyr"] = pd.to_numeric(df["fyr"], errors="coerce").astype("Int64")

    # Convert dlrsn (delisting reason) to nullable integer
    if "dlrsn" in df.columns:
        df["dlrsn"] = pd.to_numeric(df["dlrsn"], errors="coerce").astype("Int64")

    # Convert numeric fields
    numeric_cols = [
        "act", "lct", "at", "lt",
        "seq", "teq", "ceq",
        "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp",
        "ebit", "ebitda", "xint",
        "capx", "oancf", "fincf", "ivncf", "wcapch",
        "re", "pstk", "opincar",
        "csho", "mkvalt", "prcc_f", "bkvlps",
        "nipfc",
    ]
    numeric_cols = [c for c in numeric_cols if c in df.columns]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# =========================
# 3. Compute financial KPIs
# =========================

def compute_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a selected set of key financial KPIs from Compustat data.
    Also create a *raw* bankruptcy indicator from dlrsn (bankruptcy-related delisting).
    The final 'distress' label with horizon (T+1, T+2, ...) will be built in a later notebook.
    """

    d = df.copy()

    # -------- Raw bankruptcy label from dlrsn --------
    # dlrsn = 2 or 3 → bankruptcy / liquidation (raw label, no horizon yet)
    if "dlrsn" in d.columns:
        d["bankruptcy_dlrsn"] = d["dlrsn"].isin([2, 3]).astype(int)
    else:
        d["bankruptcy_dlrsn"] = None  # Should never happen if pipeline is correct

    # =================================================
    # KPI computations
    # =================================================

    # Net income
    if "nipfc" in d.columns:
        d["net_income"] = d["nipfc"]

    # Total debt = DLTT + DLC
    if {"dltt", "dlc"}.issubset(d.columns):
        d["total_debt"] = d["dltt"] + d["dlc"]

    # EBT = EBIT - interest expense
    if {"ebit", "xint"}.issubset(d.columns):
        d["ebt"] = d["ebit"] - d["xint"]

    # Choose equity proxy: seq → ceq → teq
    equity_seq = None
    for eq in ["seq", "ceq", "teq"]:
        if eq in d.columns:
            equity_seq = d[eq]
            break

    # ---------- Profitability ----------
    if {"ebt", "at"}.issubset(d.columns):
        d["roa"] = d["ebt"] / d["at"]

    if {"ebitda", "revt"}.issubset(d.columns):
        d["ebitda_margin"] = d["ebitda"] / d["revt"]

    # ---------- Leverage ----------
    if {"lt", "at"}.issubset(d.columns):
        d["debt_ratio"] = d["lt"] / d["at"]

    if "total_debt" in d.columns and equity_seq is not None:
        d["total_debt_to_equity"] = d["total_debt"] / equity_seq

    # ---------- Liquidity ----------
    if {"act", "lct"}.issubset(d.columns):
        d["current_ratio"] = d["act"] / d["lct"]

    if {"che", "lct"}.issubset(d.columns):
        d["cash_ratio"] = d["che"] / d["lct"]

    # ---------- Coverage ----------
    if {"ebit", "xint"}.issubset(d.columns):
        d["interest_coverage"] = d["ebit"] / d["xint"].abs()

    # ---------- Cash flow ----------
    if {"oancf", "revt"}.issubset(d.columns):
        d["cfo_margin"] = d["oancf"] / d["revt"]

    if {"oancf", "capx"}.issubset(d.columns):
        d["free_cash_flow"] = d["oancf"] - d["capx"]

    # ---------- Efficiency ----------
    if {"revt", "at"}.issubset(d.columns):
        d["asset_turnover"] = d["revt"] / d["at"]

    # ---------- Growth ----------
    if {"gvkey", "fyear"}.issubset(d.columns):
        d = d.sort_values(["gvkey", "fyear"])
        if "revt" in d.columns:
            d["sales_growth"] = d.groupby("gvkey")["revt"].pct_change()
        if "at" in d.columns:
            d["asset_growth"] = d.groupby("gvkey")["at"].pct_change()

    # ---------- Distress-style ratios ----------
    if {"act", "lct", "at"}.issubset(d.columns):
        d["working_capital_to_assets"] = (d["act"] - d["lct"]) / d["at"]

    if {"re", "at"}.issubset(d.columns):
        d["retained_earnings_to_assets"] = d["re"] / d["at"]

    # ---------- Market ratios ----------
    equity_for_bm = None
    for eq in ["ceq", "seq"]:
        if eq in d.columns:
            equity_for_bm = d[eq]
            break

    if equity_for_bm is not None and "mkvalt" in d.columns:
        d["book_to_market"] = equity_for_bm / d["mkvalt"]

    if {"prcc_f", "bkvlps"}.issubset(d.columns):
        d["price_to_book"] = d["prcc_f"] / d["bkvlps"]

    return d


# =========================
# 4. Main execution script
# =========================

if __name__ == "__main__":
    path_input = (
        "/files/financial-kpis-analysis-and-distress-prediction/"
        "data/raw/compustat_data.csv"
    )
    path_output = (
        "/files/financial-kpis-analysis-and-distress-prediction/"
        "data/processed/compustat_kpis.csv"
    )

    os.makedirs(os.path.dirname(path_output), exist_ok=True)

    df_raw = load_compustat_data(path_input)
    df_prep = preprocess_compustat(df_raw)
    df_kpis = compute_kpis(df_prep)

    df_kpis.to_csv(path_output, index=False)
    print(f"KPIs + raw bankruptcy label saved to: {path_output}")


/tmp/ipykernel_484/2069348467.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["datadate"] = pd.to_datetime(df["datadate"], errors="coerce")
/tmp/ipykernel_484/2069348467.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["fyear"] = pd.to_numeric(df["fyear"], errors="coerce").astype("Int64")
/tmp/ipykernel_484/2069348467.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

KPIs + raw bankruptcy label saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/compustat_kpis.csv
